In [1]:
import ast
s = "['dodge2022','strubell2019']"
x = ast.literal_eval('"' + s + '"')

In [1]:
# %%
import json
import numpy as np
import pandas as pd
import ast
import re
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util
from tqdm.notebook import tqdm

# %%
# 1. Load Data
print("Loading data...")
with open('../index.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

train_df = pd.read_csv('../data/train_QA.csv')
questions = train_df['question'].to_list()

# Ground Truth Parsing
raw_ref_ids = train_df['ref_id'].to_list()
raw_ref_ids = ["['" + e + "']" if e[0]!='[' else e for e in raw_ref_ids]
gt_ref_ids = [ast.literal_eval(e.replace('"', '')) for e in raw_ref_ids]

# Document Preparation
doc_ids = []
documents = []
doc_id_map = {}

for idx, item in enumerate(data):
    doc_id = str(item['id'])
    text_content = item['text']
    doc_ids.append(doc_id)
    documents.append(text_content)
    doc_id_map[idx] = doc_id

print(f"Data Loaded. Questions: {len(questions)}, Documents: {len(documents)}")

# %%
# 2. Define Tokenizers
def optimized_tokenizer(text):
    # Remove punctuation, cleaner split
    return re.findall(r'\w+', text.lower())

def default_tokenizer(text):
    # Standard whitespace split (Baseline behavior)
    return text.lower().split()

# %%
# 3. Define Strict Recall Function
def calculate_strict_metrics(name, retrieved_indices_list, k_list=[1, 5, 10, 20, 50]):
    hits = {k: 0 for k in k_list}
    valid_count = 0
    
    for i, top_indices in enumerate(retrieved_indices_list):
        target_ids = set(gt_ref_ids[i])
        if not target_ids or list(target_ids)[0] == 'is_blank':
            continue
        
        valid_count += 1
        
        # Parse IDs (e.g. 15_0 -> 15)
        target_base = set([e.split('_')[0] for e in target_ids])
        retrieved_base = [doc_id_map[idx].split('_')[0] for idx in top_indices]
        
        for k in k_list:
            top_k_set = set(retrieved_base[:k])
            # Strict Recall: Target must be a SUBSET of Retrieved (All refs must be found)
            if target_base.issubset(top_k_set):
                hits[k] += 1
                
    metrics = {k: hits[k]/valid_count for k in k_list}
    return metrics

# Dictionary to store all results for final table generation
all_metrics = {}

# %%
# 4. TF-IDF Optimized
# Params from Grid Search: ngram=(1, 1), min_df=2, sublinear_tf=False, max_df=0.95
print("Running TF-IDF (Optimized)...")
vectorizer_opt = TfidfVectorizer(
    tokenizer=optimized_tokenizer, token_pattern=None,
    ngram_range=(1, 1), min_df=2, sublinear_tf=False, norm='l2', max_df=0.95
)
tfidf_mat_opt = vectorizer_opt.fit_transform(documents)
query_mat_opt = vectorizer_opt.transform(questions)
from sklearn.metrics.pairwise import cosine_similarity
scores_opt = cosine_similarity(query_mat_opt, tfidf_mat_opt)

indices_opt = [np.argsort(scores_opt[i])[::-1][:50] for i in range(len(questions))]
all_metrics['TF-IDF (Optimized)'] = calculate_strict_metrics("TF-IDF (Optimized)", indices_opt)

# %%
# 5. BM25 Optimized
# Params from Grid Search: k1=1.2, b=0.75
print("Running BM25 (Optimized)...")
tokenized_corpus_opt = [optimized_tokenizer(doc) for doc in documents]
bm25_opt = BM25Okapi(tokenized_corpus_opt, k1=1.2, b=0.75)

indices_bm25_opt = []
for q in questions:
    scores = bm25_opt.get_scores(optimized_tokenizer(q))
    indices_bm25_opt.append(np.argsort(scores)[::-1][:50])
all_metrics['BM25 (Optimized)'] = calculate_strict_metrics("BM25 (Optimized)", indices_bm25_opt)

# %%
# 6. BM25 Baseline
# Defaults: k1=1.5, b=0.75, Default tokenizer
print("Running BM25 (Baseline)...")
tokenized_corpus_base = [default_tokenizer(doc) for doc in documents]
bm25_base = BM25Okapi(tokenized_corpus_base, k1=1.5, b=0.75)

indices_bm25_base = []
for q in questions:
    scores = bm25_base.get_scores(default_tokenizer(q))
    indices_bm25_base.append(np.argsort(scores)[::-1][:50])
all_metrics['BM25 (Baseline)'] = calculate_strict_metrics("BM25 (Baseline)", indices_bm25_base)

# %%
# 7. TF-IDF Baseline
# Defaults: Default tokenizer (via analyzer), default params
print("Running TF-IDF (Baseline)...")
vectorizer_base = TfidfVectorizer(tokenizer=default_tokenizer, token_pattern=None)
tfidf_mat_base = vectorizer_base.fit_transform(documents)
query_mat_base = vectorizer_base.transform(questions)
scores_base = cosine_similarity(query_mat_base, tfidf_mat_base)

indices_base = [np.argsort(scores_base[i])[::-1][:50] for i in range(len(questions))]
all_metrics['TF-IDF (Baseline)'] = calculate_strict_metrics("TF-IDF (Baseline)", indices_base)

# %%
# 8. Dense Retrieval (BGE-M3)
print("Running BGE-M3 Dense...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_bge = SentenceTransformer('BAAI/bge-m3', device=device)

doc_embs_bge = model_bge.encode(documents, batch_size=8, show_progress_bar=True, convert_to_tensor=True)
q_embs_bge = model_bge.encode(questions, batch_size=8, show_progress_bar=True, convert_to_tensor=True)

hits_bge = util.semantic_search(q_embs_bge, doc_embs_bge, top_k=50)
indices_bge = [[res['corpus_id'] for res in hit] for hit in hits_bge]
all_metrics['BGE-M3-Dense'] = calculate_strict_metrics("BGE-M3-Dense", indices_bge)

# %%
# 9. Dense Retrieval (Nomic v1.5)
print("Running Nomic v1.5...")
model_nomic = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', trust_remote_code=True, device=device)

nomic_docs = ['search_document: ' + d for d in documents]
nomic_qs = ['search_query: ' + q for q in questions]

doc_embs_nomic = model_nomic.encode(nomic_docs, batch_size=8, show_progress_bar=True, convert_to_tensor=True)
q_embs_nomic = model_nomic.encode(nomic_qs, batch_size=8, show_progress_bar=True, convert_to_tensor=True)

hits_nomic = util.semantic_search(q_embs_nomic, doc_embs_nomic, top_k=50)
indices_nomic = [[res['corpus_id'] for res in hit] for hit in hits_nomic]
all_metrics['Nomic-v1.5'] = calculate_strict_metrics("Nomic-v1.5", indices_nomic)

# %%
# 10. Generate Markdown Table with Bolding
print("\n### Recall Metrics Comparison")
print("| Model | Recall@1 | Recall@5 | Recall@10 | Recall@20 | Recall@50 |")
print("| :--- | :---: | :---: | :---: | :---: | :---: |")

# Order of rows
row_order = [
    'TF-IDF (Optimized)',
    'BM25 (Optimized)',
    'BM25 (Baseline)',
    'TF-IDF (Baseline)',
    'BGE-M3-Dense',
    'Nomic-v1.5'
]

# Find max values per column for bolding
max_vals = {k: 0.0 for k in [1, 5, 10, 20, 50]}
for model_name, metrics in all_metrics.items():
    for k in [1, 5, 10, 20, 50]:
        if metrics[k] > max_vals[k]:
            max_vals[k] = metrics[k]

# Print rows
for name in row_order:
    if name in all_metrics:
        cols = []
        for k in [1, 5, 10, 20, 50]:
            val = all_metrics[name][k]
            # Check if this value is the max (allow slight float tolerance)
            if val >= max_vals[k] - 1e-9:
                cols.append(f"**{val:.3f}**")
            else:
                cols.append(f"{val:.3f}")
        
        print(f"| {name} | {cols[0]} | {cols[1]} | {cols[2]} | {cols[3]} | {cols[4]} |")


Loading data...
Data Loaded. Questions: 41, Documents: 639
Running TF-IDF (Optimized)...
Running BM25 (Optimized)...
Running BM25 (Baseline)...
Running TF-IDF (Baseline)...
Running BGE-M3 Dense...


Batches:   0%|          | 0/80 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Running Nomic v1.5...


<All keys matched successfully>


Batches:   0%|          | 0/80 [00:00<?, ?it/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]


### Recall Metrics Comparison
| Model | Recall@1 | Recall@5 | Recall@10 | Recall@20 | Recall@50 |
| :--- | :---: | :---: | :---: | :---: | :---: |
| TF-IDF (Optimized) | 0.744 | **0.923** | 0.923 | **0.974** | **1.000** |
| BM25 (Optimized) | **0.769** | **0.923** | 0.923 | 0.949 | 0.974 |
| BM25 (Baseline) | 0.744 | 0.897 | 0.923 | 0.949 | **1.000** |
| TF-IDF (Baseline) | 0.718 | 0.872 | 0.923 | 0.949 | **1.000** |
| BGE-M3-Dense | 0.615 | **0.923** | **0.974** | **0.974** | **1.000** |
| Nomic-v1.5 | 0.590 | 0.795 | 0.923 | 0.949 | **1.000** |
